# EWS Fraud Detection - Tahap 6B: Integrasi Fitur Tekstual & Penggabungan Dataset

Notebook ini bertujuan untuk mengintegrasikan hasil ekstraksi NLP Baseline dari Laporan Tahunan (`Annual_Report_Features.xlsx`) dengan data keuangan hasil skoring (`Fraud_Dataset_Scored.xlsx`).

Kita akan:
1. Membersihkan duplikasi data laporan tahunan dengan agregasi rata-rata.
2. Melakukan standardisasi skala (Min-Max 0-100) pada fitur tekstual (`sentiment`, `risk_words`, `readability`).
3. Menghitung **Narrative Risk Score** dengan bobot: **40% Sentiment + 30% Risk Words + 30% Readability**.
4. Melakukan Left Join antara data keuangan berskor (**Financial Risk Score**) dengan Narrative Risk Score.
5. Menghitung **Combined Fraud Score** dengan bobot: **70% Financial Risk Score + 30% Narrative Risk Score**.
6. Menyimpan hasil akhir ke `Final_EWS_Dataset.xlsx`.

In [8]:
import pandas as pd
import os
import numpy as np

features_path = "Annual_Report_Features.xlsx"
scored_path = "Fraud_Dataset_Scored.xlsx"
final_output_path = "Final_EWS_Dataset.xlsx"

## 1. Memuat & Menganalisis Duplikasi Laporan Tahunan

In [9]:
if os.path.exists(features_path):
    df_features = pd.read_excel(features_path)
    print(f"Dataset Laporan Tahunan berhasil dimuat. Shape asli: {df_features.shape}")
else:
    print("[ERROR] Berkas Annual_Report_Features.xlsx tidak ditemukan!")

Dataset Laporan Tahunan berhasil dimuat. Shape asli: (5397, 9)


In [10]:
# 1. Cari Berapa Kombinasi Unik
print("Kombinasi Unik (Kode, tahun) Shape:")
print(df_features[["Kode","tahun"]].drop_duplicates().shape)

# 2. Cari berapa yang duplikat
print("\nJumlah Duplikat (Kode, tahun) Sum:")
print(df_features.duplicated(subset=["Kode","tahun"]).sum())

Kombinasi Unik (Kode, tahun) Shape:
(2336, 2)

Jumlah Duplikat (Kode, tahun) Sum:
3061


## 2. Agregasi Mean untuk Mengeliminasi Duplikasi

In [11]:
# 3. Jikalau memang duplikat ada, lakukan agregasi rata-rata
annual_features = (
    df_features.groupby(["Kode","tahun"])
    .agg({
        "sentiment":"mean",
        "risk_words":"mean",
        "readability":"mean",
        "text_length":"mean"
    })
    .reset_index()
)

print(f"Shape setelah agregasi: {annual_features.shape}")
display(annual_features.head())

Shape setelah agregasi: (2336, 6)


,Kode,tahun,sentiment,risk_words,readability,text_length
0,AALI,2022,-0.281046,77.0,60.672312,311903.0
1,AALI,2023,-0.155660,46.0,30.758923,155282.0
2,AALI,2024,-0.119512,48.0,30.579044,162500.0
3,ABBA,2021,0.276873,66.0,55.890944,407540.0
4,ABBA,2022,0.211538,50.0,56.894978,436983.0


## 3. Penggabungan dengan Dataset Keuangan Ter-Skor

In [12]:
if os.path.exists(scored_path):
    df_scored = pd.read_excel(scored_path)
    print(f"Dataset skoring keuangan berhasil dimuat. Shape asli: {df_scored.shape}")
else:
    print("[ERROR] Berkas Fraud_Dataset_Scored.xlsx tidak ditemukan!")

Dataset skoring keuangan berhasil dimuat. Shape asli: (1637, 69)


In [13]:
# Standardisasi key penggabungan
df_scored['kode_clean'] = df_scored['kode'].astype(str).str.strip().str.upper()
df_scored['tahun_clean'] = df_scored['tahun'].astype(int)

annual_features['kode_clean'] = annual_features['Kode'].astype(str).str.strip().str.upper()
annual_features['tahun_clean'] = annual_features['tahun'].astype(int)

# Jalankan Left Join
df_final = pd.merge(
    df_scored,
    annual_features[['kode_clean', 'tahun_clean', 'sentiment', 'risk_words', 'readability', 'text_length']],
    on=['kode_clean', 'tahun_clean'],
    how='left'
)

# Hapus kolom pembersihan temporer
df_final = df_final.drop(columns=['kode_clean', 'tahun_clean'])
print(f"Shape setelah penggabungan: {df_final.shape}")

Shape setelah penggabungan: (1637, 73)


## 4. Standardisasi Fitur Laporan Tahunan & Perhitungan Narrative Risk Score
Kita melakukan standardisasi Min-Max (skala 0 - 100) pada ketiga fitur tekstual:
- **Sentiment**: Dibalik (inversi) karena sentimen yang lebih rendah/negatif menunjukkan risiko yang lebih tinggi.
- **Risk Words**: Normal (nilai lebih tinggi menunjukkan risiko lebih tinggi).
- **Readability**: Normal (nilai indeks LIX lebih tinggi menunjukkan teks lebih sulit dibaca / potensi obfuscation yang berisiko tinggi).

Rumus **Narrative Risk Score**:
$$\text{Narrative Risk Score} = 40\% \times \text{Sentiment Scaled} + 30\% \times \text{Risk Words Scaled} + 30\% \times \text{Readability Scaled}$$

In [14]:
# Standardisasi hanya untuk baris yang memiliki data NLP asli (non-NaN)
sentiment_min = df_final['sentiment'].min()
sentiment_max = df_final['sentiment'].max()
df_final['sentiment_scaled'] = (sentiment_max - df_final['sentiment']) / (sentiment_max - sentiment_min) * 100

risk_words_min = df_final['risk_words'].min()
risk_words_max = df_final['risk_words'].max()
df_final['risk_words_scaled'] = (df_final['risk_words'] - risk_words_min) / (risk_words_max - risk_words_min) * 100

readability_min = df_final['readability'].min()
readability_max = df_final['readability'].max()
df_final['readability_scaled'] = (df_final['readability'] - readability_min) / (readability_max - readability_min) * 100

# Hitung Narrative Risk Score
df_final['narrative_risk_score'] = (
    0.40 * df_final['sentiment_scaled'] +
    0.30 * df_final['risk_words_scaled'] +
    0.30 * df_final['readability_scaled']
)

## 5. Perhitungan Combined Fraud Risk Score
1. **Financial Risk Score**: Menggunakan nilai dari kolom skoring keuangan lama (`fraud_score`).
2. **Narrative Risk Score**: Mengisi baris kosong (akibat laporan tahunan absen) dengan nilai `financial_risk_score` agar tidak membuang atau merendahkan secara buatan status risiko emiten tersebut.
3. **Combined Fraud Score**:
$$\text{Combined Fraud Score} = 70\% \times \text{Financial Risk Score} + 30\% \times \text{Narrative Risk Score}$$
4. **Combined Fraud Status**:
   - $0 - 30$: Low
   - $31 - 60$: Medium
   - $61 - 80$: High
   - $> 80$: Critical

In [15]:
# Salin/rename financial risk score
df_final['financial_risk_score'] = df_final['fraud_score']

# Penanganan missing value secara fair
df_final['narrative_risk_score'] = df_final['narrative_risk_score'].fillna(df_final['financial_risk_score'])

# Hitung Combined Fraud Score
df_final['combined_fraud_score'] = 0.70 * df_final['financial_risk_score'] + 0.30 * df_final['narrative_risk_score']

# Klasifikasikan
def classify_combined_status(score):
    if score <= 30:
        return 'Low'
    elif score <= 60:
        return 'Medium'
    elif score <= 80:
        return 'High'
    else:
        return 'Critical'

df_final['combined_fraud_status'] = df_final['combined_fraud_score'].apply(classify_combined_status)

print("Distribusi Combined Fraud Status:")
print(df_final['combined_fraud_status'].value_counts())

# Isi nilai mentah NLP yang NaN dengan 0 untuk kemudahan visualisasi/dashboard
for col in ['sentiment', 'risk_words', 'readability', 'text_length']:
    df_final[col] = df_final[col].fillna(0.0)

# Simpan berkas hasil akhir
df_final.to_excel(final_output_path, index=False)
print(f"\nDataset akhir EWS berhasil disimpan ke: {final_output_path} dengan shape: {df_final.shape}")

Distribusi Combined Fraud Status:
combined_fraud_status
Low       1426
Medium     200
High        11
Name: count, dtype: int64

Dataset akhir EWS berhasil disimpan ke: Final_EWS_Dataset.xlsx dengan shape: (1637, 80)
